# Bônus — Data contract (`movies`)
Demonstra o contrato declarado em
`config/contracts/movies_contract.yaml` gerando checks Spark e o
validator `$jsonSchema` do MongoDB a partir do MESMO arquivo
(Aula 4, seção 4.4). Roda **depois** de `01_run_pipeline.py`, já que
depende da tabela `meu_catalog.bronze.sample_mflix__movies` existir.

In [0]:
import sys
import json
sys.path.append("../src")

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, DoubleType, IntegerType, StringType, StructField, StructType

from ingestion.contract import apply_checks, generate_mongo_validator, generate_spark_checks, load_contract

contrato = load_contract("../config/contracts/movies_contract.yaml")
checks = generate_spark_checks(contrato)
for nome, expr in checks:
    print(f"{nome:24} -> {expr}")

## Validator equivalente para impor o mesmo contrato na origem (MongoDB)

In [0]:
print(json.dumps(generate_mongo_validator(contrato), indent=2, ensure_ascii=False))

## Aplicando os checks ao lote ingerido
A Bronze genérica guarda o documento como JSON bruto (`body`) — para
validar o contrato, fazemos o parse pontual **apenas aqui**, sem
alterar o formato de gravação da Bronze (que continua fiel à origem,
R6). Este é exatamente o tipo de tipagem que migraria para a camada
Silver em uma evolução futura do projeto.

In [ ]:
schema_movies = StructType([
    StructField("_id", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", ArrayType(StringType()), True),
    StructField("runtime", IntegerType(), True),
    StructField("imdb", StructType([
        StructField("rating", DoubleType(), True),
    ]), True),
])

df_bronze = spark.table("meu_catalog.bronze.sample_mflix__movies")
df_parsed = (
    df_bronze
    .select("_source_id", F.from_json("body", schema_movies).alias("doc"))
    .select("_source_id", "doc.*")
)

resultados = apply_checks(df_parsed, checks)
spark.createDataFrame(resultados).display()

## Alterando o contrato sem tocar no código
Exemplo: torne `runtime` obrigatório em
`config/contracts/movies_contract.yaml` (`obrigatorio: true`) e rode
esta célula de novo — o veredito muda sem alterar uma linha de
código Python.

In [0]:
contrato_recarregado = load_contract("../config/contracts/movies_contract.yaml")
novos_checks = generate_spark_checks(contrato_recarregado)
spark.createDataFrame(apply_checks(df_parsed, novos_checks)).display()